# An unfamiliar dataframe

Discover a field family, inspect an exception with duplicate indexes, choose a census, refine a scope, and save a recipe.

In [ ]:
import pandas as pd
from IPython.display import SVG, display

import fieldwork as fw

df = pd.DataFrame(
    {
        "site": ["N", "N", "S", "S"],
        "exam": [1, 1, 2, 2],
        "image_1": [1, 2, 3, 4],
        "image_2": [5, 6, 7, None],
    },
    index=[0] * 4,
)

In [ ]:
overview = fw.explore(df)
print(overview)
availability = fw.missingness(df, entity="exam", min_implication=0.75)
display(SVG(fw.render_svg(availability)))

In [ ]:
edge = next(
    f
    for f in availability["findings"]
    if f["pattern"] == "presence_implication"
    and [c["column"] for c in f["features"]] == ["image_1", "image_2"]
)
exceptions = availability.inspect(df, edge["id"], exceptions=True)
assert len(exceptions) == 1
exceptions

In [ ]:
paths = fw.suggest_paths(df, features=["site", "exam"])
tree = paths.best.census(df)
display(SVG(fw.render_svg(tree)))

In [ ]:
signature = next(s for s in availability["signatures"] if "image_2" in s["absent"])
scope = availability.select(df, signature["finding_id"], name="missing second image")
local = fw.missingness(df, scope=scope)
comparison = fw.compare(availability, local)
paths = fw.suggest_paths(df, scope=scope, start_with=["site"], missing={"image_2": [-999]})
assert paths.best.census(df)["scopes"][0]["input_rows"] == len(df)
comparison.to_frame("changes")

In [ ]:
recipe = fw.Recipe(
    "missingness",
    {"entity": "exam", "min_implication": 0.75},
    notes="Check image pairs in each delivery",
)
assert recipe.run(df)["availability"] == availability["availability"]
# recipe.save("availability-recipe.json")
# fw.Recipe.load("availability-recipe.json").run(next_delivery)

In [ ]:
balanced = fw.missingness(df, entity="exam", unit="entities")
assert balanced["analysis_unit"]["denominator"] == 2
connections = overview.relationships("image_1", kinds=["indexed_name"])
finding_id = connections.iloc[0]["evidence.overview_finding_id"]
overview.inspect(df, finding_id)